# 04 - Impacto Normativo y Riesgos para Salud Pública

Objetivo: convertir el análisis técnico en un relato de **riesgo real**, respaldado con **normativa china (GB 3095)** y evidencia de salud pública.

> Mensaje central: el histórico 2013-2017 muestra mejora en tendencia, pero una brecha crítica de cumplimiento diario/anual frente a estándares actuales y más aún frente a guías sanitarias.

## Índice

1. Marco regulatorio (China + salud pública)
2. Carga histórica y tendencia
3. Incumplimiento diario (riesgo operativo)
4. Hotspots territoriales
5. Correlaciones y señales causales útiles
6. Mensajes de impacto para presentación
7. Matriz de respuesta alineada a norma

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

candidate_roots = [Path('.').resolve(), Path('..').resolve()]
project_root = next((p for p in candidate_roots if (p / 'data').exists() and (p / 'notebooks').exists()), None)
if project_root is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto.')

data_path = project_root / 'data' / 'processed' / 'beijing_unified_cleaned.csv'
reports_path = project_root / 'reports'
reports_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path, parse_dates=['date'])
df = df.sort_values(['station', 'date'])

print(f"Registros: {len(df):,}")
print(f"Estaciones: {df['station'].nunique()}")
print(f"Rango temporal: {df['date'].min()} -> {df['date'].max()}")

Registros: 420,768
Estaciones: 12
Rango temporal: 2013-03-01 00:00:00 -> 2017-02-28 23:00:00


## 1) Marco regulatorio y sanitario

### Referencia normativa usada

- **GB 3095-2012** (Grade II, histórico): PM2.5 anual 35, PM2.5 24h 75, PM10 anual 70, PM10 24h 150.
- **GB 3095-2026 (fase 1, 2026-2030)**: PM2.5 anual 30, PM2.5 24h 60, PM10 anual 60, PM10 24h 120.
- **GB 3095-2026 (fase posterior)**: PM2.5 anual 25 y PM2.5 24h 50.

### Contexto sanitario

Las guías OMS son más estrictas (PM2.5 anual 5, PM2.5 24h 15), por lo que cumplir norma local no implica eliminar riesgo sanitario.

In [ ]:
thresholds = {
    'pm25_annual': {'GB3095_2012_II': 35, 'GB3095_2026_phase1_II': 30, 'GB3095_2026_phase2_II': 25, 'WHO_AQG_2021': 5},
    'pm25_daily': {'GB3095_2012_II': 75, 'GB3095_2026_phase1_II': 60, 'GB3095_2026_phase2_II': 50, 'WHO_AQG_2021': 15},
    'pm10_annual': {'GB3095_2012_II': 70, 'GB3095_2026_phase1_II': 60},
    'pm10_daily': {'GB3095_2012_II': 150, 'GB3095_2026_phase1_II': 120}
}

for k, v in thresholds.items():
    print('\n', k)
    print(pd.Series(v))


 pm25_annual
GB3095_2012_II           35
GB3095_2026_phase1_II    30
GB3095_2026_phase2_II    25
WHO_AQG_2021              5
dtype: int64

 pm25_daily
GB3095_2012_II           75
GB3095_2026_phase1_II    60
GB3095_2026_phase2_II    50
WHO_AQG_2021             15
dtype: int64

 pm10_annual
GB3095_2012_II           70
GB3095_2026_phase1_II    60
dtype: int64

 pm10_daily
GB3095_2012_II           150
GB3095_2026_phase1_II    120
dtype: int64


## 2) Peligros principales para salud y ciudad

1. Exposición crónica elevada (riesgo cardiopulmonar acumulado).
2. Episodios agudos frecuentes (presión en urgencias y productividad).
3. Riesgo desigual por territorio (hotspots urbanos).
4. Vulnerabilidad estacional (invierno/estabilidad atmosférica).
5. Riesgo reputacional y regulatorio si no se adapta la gestión al estándar 2026.

In [ ]:
# Derivados temporales
raw = df.copy()
raw['year'] = raw['date'].dt.year
raw['month'] = raw['date'].dt.month
raw['hour'] = raw['date'].dt.hour

# Serie anual ciudad
city_year = raw.groupby('year', as_index=False).agg(
    pm25_mean=('PM2.5', 'mean'),
    pm10_mean=('PM10', 'mean'),
    rows=('PM2.5', 'size')
)

# Años completos para comparación justa
full_years = city_year[city_year['year'].between(2014, 2016)].copy()

# Serie diaria por estación
station_day = raw.groupby(['station', raw['date'].dt.date], as_index=False).agg(
    pm25_day=('PM2.5', 'mean'),
    pm10_day=('PM10', 'mean')
).rename(columns={'date': 'day'})

# Cumplimiento diario
station_day['pm25_gt75'] = station_day['pm25_day'] > 75
station_day['pm25_gt60'] = station_day['pm25_day'] > 60
station_day['pm25_gt50'] = station_day['pm25_day'] > 50
station_day['pm10_gt150'] = station_day['pm10_day'] > 150
station_day['pm10_gt120'] = station_day['pm10_day'] > 120

# Cumplimiento anual por estación (años completos)
station_year = raw[raw['year'].between(2014, 2016)].groupby(['station', 'year'], as_index=False).agg(
    pm25_mean=('PM2.5', 'mean'),
    pm10_mean=('PM10', 'mean')
)

summary = {
    'rows': len(raw),
    'stations': raw['station'].nunique(),
    'pm25_global_mean': raw['PM2.5'].mean(),
    'pm25_hour_gt35_pct': (raw['PM2.5'] > 35).mean() * 100,
    'pm25_hour_gt75_pct': (raw['PM2.5'] > 75).mean() * 100,
    'pm25_day_gt60_pct': station_day['pm25_gt60'].mean() * 100,
    'pm25_day_gt50_pct': station_day['pm25_gt50'].mean() * 100,
}

summary

In [ ]:
# Gráfico: PM2.5 anual ciudad vs umbrales
plt.figure(figsize=(10, 5))
plt.plot(city_year['year'], city_year['pm25_mean'], marker='o', color='darkred', label='PM2.5 medio anual (dataset)')

plt.axhline(35, color='orange', linestyle='--', label='GB3095-2012 II (35)')
plt.axhline(30, color='royalblue', linestyle='--', label='GB3095-2026 Fase 1 II (30)')
plt.axhline(25, color='navy', linestyle='--', label='GB3095-2026 Fase 2 II (25)')
plt.axhline(5, color='green', linestyle=':', label='OMS AQG 2021 (5)')

plt.title('PM2.5 anual vs umbrales regulatorios/sanitarios')
plt.xlabel('Año')
plt.ylabel('PM2.5 (µg/m³)')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(reports_path / '10_normativa_pm25_anual_vs_umbral.png', dpi=150)
plt.show()

city_year

In [ ]:
# Barras de incumplimiento diario
daily_kpi = pd.DataFrame({
    'Indicador': ['PM2.5 > 75', 'PM2.5 > 60', 'PM2.5 > 50', 'PM10 > 150', 'PM10 > 120'],
    'Pct': [
        station_day['pm25_gt75'].mean() * 100,
        station_day['pm25_gt60'].mean() * 100,
        station_day['pm25_gt50'].mean() * 100,
        station_day['pm10_gt150'].mean() * 100,
        station_day['pm10_gt120'].mean() * 100,
    ]
})

plt.figure(figsize=(10, 5))
sns.barplot(data=daily_kpi, x='Indicador', y='Pct', palette='Reds_r')
plt.title('Incumplimiento diario por umbral regulatorio')
plt.ylabel('% de estación-día en superación')
plt.xlabel('')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(reports_path / '11_normativa_excedencias_diarias.png', dpi=150)
plt.show()

daily_kpi

In [ ]:
# Tendencia anual del incumplimiento diario PM2.5
station_day['year'] = pd.to_datetime(station_day['day']).dt.year
yearly_exceed = station_day.groupby('year', as_index=False).agg(
    pm25_gt75=('pm25_gt75', 'mean'),
    pm25_gt60=('pm25_gt60', 'mean'),
    pm25_gt50=('pm25_gt50', 'mean')
)

for c in ['pm25_gt75', 'pm25_gt60', 'pm25_gt50']:
    yearly_exceed[c] = yearly_exceed[c] * 100

plt.figure(figsize=(10, 5))
plt.plot(yearly_exceed['year'], yearly_exceed['pm25_gt75'], marker='o', label='>75 (GB2012)')
plt.plot(yearly_exceed['year'], yearly_exceed['pm25_gt60'], marker='o', label='>60 (GB2026 fase 1)')
plt.plot(yearly_exceed['year'], yearly_exceed['pm25_gt50'], marker='o', label='>50 (GB2026 fase 2)')
plt.title('Evolución anual del incumplimiento diario PM2.5')
plt.ylabel('% de estación-día en superación')
plt.xlabel('Año')
plt.legend()
plt.tight_layout()
plt.savefig(reports_path / '12_normativa_tendencia_incumplimiento.png', dpi=150)
plt.show()

yearly_exceed

In [ ]:
# Hotspots: estaciones con más días PM2.5 > 60
hotspots = station_day.groupby('station', as_index=False).agg(
    exc60_rate=('pm25_gt60', 'mean'),
    exc50_rate=('pm25_gt50', 'mean'),
    exc75_rate=('pm25_gt75', 'mean')
)
for c in ['exc60_rate', 'exc50_rate', 'exc75_rate']:
    hotspots[c] = hotspots[c] * 100

hotspots = hotspots.sort_values('exc60_rate', ascending=False)

plt.figure(figsize=(11, 5))
sns.barplot(data=hotspots, x='station', y='exc60_rate', color='firebrick')
plt.title('Hotspots por frecuencia de superación PM2.5 > 60 (GB2026 fase 1)')
plt.ylabel('% de estación-día')
plt.xlabel('Estación')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(reports_path / '13_normativa_hotspots_gt60.png', dpi=150)
plt.show()

hotspots.head(12)

In [ ]:
# Correlaciones con PM2.5
corr_cols = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
corr = raw[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlación entre PM2.5 y variables explicativas')
plt.tight_layout()
plt.savefig(reports_path / '14_normativa_correlaciones_pm25.png', dpi=150)
plt.show()

corr['PM2.5'].sort_values(ascending=False)

## 6) Mensajes de alto impacto para presentación

- Más de la mitad de los estación-día superan PM2.5 >60 (fase 1 de la nueva norma).
- El riesgo no es homogéneo: hay hotspots claros con carga repetida.
- La mejora de tendencia no es suficiente para cumplimiento pleno anual en este histórico.
- El viento aparece como palanca física relevante (mejor dispersión, menor PM2.5).
- La gestión debe ser multipolutante y no solo centrada en PM2.5.

In [ ]:
# Frases automáticas para slide ejecutiva
msg = {
    'pm25_mean': summary['pm25_global_mean'],
    'pm25_gt60_day': summary['pm25_day_gt60_pct'],
    'pm25_gt50_day': summary['pm25_day_gt50_pct'],
    'worst_station': hotspots.iloc[0]['station'],
    'worst_station_rate': hotspots.iloc[0]['exc60_rate']
}

print(f"PM2.5 medio histórico: {msg['pm25_mean']:.2f} µg/m³")
print(f"Estación-día con PM2.5 >60: {msg['pm25_gt60_day']:.2f}%")
print(f"Estación-día con PM2.5 >50: {msg['pm25_gt50_day']:.2f}%")
print(f"Hotspot principal: {msg['worst_station']} ({msg['worst_station_rate']:.2f}% de días >60)")

## 7) Matriz de respuesta alineada a norma

1. Si PM2.5 diario proyectado >60: activar paquete preventivo (movilidad + fiscalización + comunicación a vulnerables).
2. Si PM2.5 diario proyectado >50 de forma persistente: activar paquete reforzado intersectorial.
3. Priorizar estaciones hotspot para intervenciones y auditoría mensual.
4. Evaluar impacto con KPIs regulatorios: % días >60, >50, >120(PM10), tendencia anual por estación.

## Referencias normativas y sanitarias

- GB 3095-2012 (MEE, versión inglesa):
  https://english.mee.gov.cn/Resources/standards/Air_Environment/quality_standard1/201605/P020160511430661307629.pdf
- Entrevista técnica oficial MEE sobre GB 3095-2026 (24-feb-2025):
  https://www.mee.gov.cn/ywdt/zbft/202502/t20250224_1100154.shtml
- Implementación oficial del estándar (20-feb-2025):
  https://www.mee.gov.cn/zcwj/gwywj/202502/t20250220_1099853.shtml
- Guía OMS AQG 2021:
  https://www.who.int/publications/i/item/9789240034228